# Fine-tune detector biển số bằng ẢNH CAMERA THẬT (Kaggle, ~3.5 tiếng)

**Vì sao:** `plate_square_best.pt` lần trước có mAP val đẹp (0.83) nhưng trên video cổng trường thật lại **bỏ sót ~65 biển** mà `plate_best.pt` bắt được, và bắt nhầm biển báo đỏ. Dataset công khai toàn ảnh biển chụp gần/rõ; camera thật thì biển chỉ ~26×20 px, mờ, nghiêng.

Notebook này fine-tune tiếp `plate_best.pt` trên: **dataset công khai (giữ đủ biển vuông + biển dài)** + **ảnh camera thật `real_plate_review`** (310 khung hình cổng trường, nhãn đã duyệt tay: 240 biển thật — 106 trong số đó cả 2 model cũ đều bỏ sót —, 132 vùng hay bị bắt nhầm làm mẫu âm, vùng không chắc đã che xám). Ảnh thật được nhân bản `REAL_REPEAT` lần cho khỏi bị dataset công khai lấn át.

**Chọn `best.pt` theo mAP trên VAL ẢNH THẬT** (3 video không có trong train) — không theo val dataset công khai.

## Cách chạy
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100). **Settings → Internet → On**.
2. **+ Add Input → Upload**: `datasets/real_plate_review.zip` (tạo bằng `python scripts/training/build_real_plate_dataset.py`) — **bắt buộc**.
3. **+ Add Input → Upload**: `models/plate_best.pt` — **nên có** (không có thì train từ `yolo11n.pt`, kém hơn nhiều).
4. *(Tùy chọn)* bộ `yolo_plate_dataset` nếu đã upload sẵn — không có thì tự tải từ Google Drive.
5. **Save Version → Save & Run All (Commit)** (chạy nền, tắt máy vẫn chạy). Không chạy interactive.
6. Xong: tab **Output** → tải `plate_real_best.pt`. Ô cuối in bảng so sánh với `plate_best.pt`.

In [ ]:
# ===== CẤU HÌNH =====
TRAIN_HOURS = 3.0          # thời gian train thực; còn ~30 phút cho cài đặt + val + xuất file
REAL_REPEAT = 6            # nhân bản ảnh thật (580 ảnh x6 ~ 3.5k) so với ~7k ảnh công khai
IMGSZ = 640                # = DETECT_WIDTH của app, đừng đổi
BATCH = 32                 # CUDA out of memory -> 16
PUBLIC_VAL_FRACTION = 0.1  # val phụ trên dataset công khai, chỉ để chắc không tụt
BASE_MODEL = ""            # trống = tự tìm *plate*.pt trong /kaggle/input, không có thì yolo11n.pt
RESUME = True              # có last.pt (phiên bị đứt) thì train tiếp
DRIVE_URL = "https://drive.google.com/uc?id=1KLK-DWgT3VoQH4fcTxAt2eB3sm7DGWAf"  # yolo_plate_dataset (8259 ảnh)

In [ ]:
import os, sys
# Kaggle cài sẵn wandb -> Ultralytics có thể đứng chờ API key trong Commit. Tắt trước khi import.
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["ULTRALYTICS_TELEMETRY"] = "False"
# Kaggle có cudf-polars-cu12 cần polars >=1.30,<1.36 — không dùng `-U polars`.
!{sys.executable} -m pip install -q --upgrade --upgrade-strategy only-if-needed ultralytics gdown "polars>=1.30,<1.36"

import torch
import ultralytics
from ultralytics import settings
for k in ("raytune", "wandb", "comet", "clearml", "mlflow", "neptune", "dvc"):
    try:
        settings.update({k: False})
    except Exception:
        pass
assert torch.cuda.is_available(), "KHÔNG có GPU: Settings -> Accelerator -> GPU T4 x2, rồi Restart Session"
print("Ultralytics", ultralytics.__version__, "| Torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

## 1. Ảnh camera thật (`real_plate_review`)

In [ ]:
import json, shutil, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def find_real():
    for root in (INPUT, WORK / "real_raw"):
        if root.exists():
            for p in root.rglob("real_plate_meta.json"):
                return p.parent
    return None

real_dir = find_real()
if real_dir is None:  # upload dạng zip mà Kaggle không tự giải nén
    zips = [p for p in INPUT.rglob("*.zip") if "real_plate" in p.name] if INPUT.exists() else []
    assert zips, "Thiếu Input real_plate_review.zip — chạy scripts/training/build_real_plate_dataset.py rồi Upload"
    zipfile.ZipFile(zips[0]).extractall(WORK / "real_raw")
    real_dir = find_real()
assert real_dir, "Không thấy real_plate_meta.json trong real_plate_review"
meta = json.loads((real_dir / "real_plate_meta.json").read_text())
print("Ảnh thật:", real_dir)
print(json.dumps(meta["stats"], indent=1))

## 2. Dataset công khai (giữ cả biển vuông lẫn biển dài) + chia val theo nhóm gốc

`rotatequandoi12`, `cropquandoi12`... là augment của `quandoi12` ⇒ luôn chung 1 split, tránh mAP val ảo.

In [ ]:
import random, re, subprocess
from collections import Counter

RAW = WORK / "raw"
IMG_EXT = (".jpg", ".jpeg", ".png")

def find_raw():
    for root in (INPUT, RAW):
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXT and p.stem.lower().startswith("xemay"):
                return p.parent
    return None

raw_dir = find_raw()
if raw_dir is None:
    zips = [p for p in INPUT.rglob("*.zip") if "real_plate" not in p.name] if INPUT.exists() else []
    if zips:
        print("Giải nén input:", zips[0])
        zipfile.ZipFile(zips[0]).extractall(RAW)
    else:
        zp = WORK / "plate_yolo.zip"
        if not zp.exists():
            subprocess.run(["gdown", DRIVE_URL, "-O", str(zp)], check=True)
        zipfile.ZipFile(zp).extractall(RAW)
    raw_dir = find_raw()
assert raw_dir, "Không tìm thấy ảnh xemay*.jpg — kiểm tra Input hoặc link Drive (Drive hết quota thì upload zip làm Input)"
print("Dataset công khai:", raw_dir)

AUG = re.compile(r"^(boder|brightness|crop|rotate)")
DST = WORK / "plate_mix"
if DST.exists():
    shutil.rmtree(DST)
for s in ("train", "val_public", "val_real"):
    (DST / "images" / s).mkdir(parents=True)
    (DST / "labels" / s).mkdir(parents=True)

pub = [p for p in sorted(raw_dir.iterdir()) if p.suffix.lower() in IMG_EXT and p.with_suffix(".txt").exists()]
groups = sorted({AUG.sub("", p.stem) for p in pub})
random.Random(42).shuffle(groups)
val_groups = set(groups[: max(1, int(len(groups) * PUBLIC_VAL_FRACTION))])
n = Counter()
for p in pub:
    rows = [l.split() for l in p.with_suffix(".txt").read_text().splitlines() if l.strip()]
    lines = "".join("0 " + " ".join(r[1:5]) + "\n" for r in rows if len(r) >= 5)  # 1 lớp 'plate'
    split = "val_public" if AUG.sub("", p.stem) in val_groups else "train"
    shutil.copy2(p, DST / "images" / split / ("pub_" + p.name))
    (DST / "labels" / split / ("pub_" + p.stem + ".txt")).write_text(lines)
    n[split] += 1

# Ảnh thật: val giữ nguyên, train nhân bản REAL_REPEAT lần (~500 MB, copy cho chạy được cả Windows)
for img in sorted((real_dir / "images" / "val").iterdir()):
    shutil.copy2(img, DST / "images" / "val_real" / img.name)
    shutil.copy2(real_dir / "labels" / "val" / (img.stem + ".txt"), DST / "labels" / "val_real" / (img.stem + ".txt"))
    n["val_real"] += 1
for img in sorted((real_dir / "images" / "train").iterdir()):
    lbl = real_dir / "labels" / "train" / (img.stem + ".txt")
    for r in range(REAL_REPEAT):
        shutil.copy2(img, DST / "images" / "train" / f"real{r}_{img.name}")
        shutil.copy2(lbl, DST / "labels" / "train" / f"real{r}_{img.stem}.txt")
        n["train_real"] += 1

def make_yaml(val):
    p = DST / f"data_{val}.yaml"
    p.write_text(f"path: {DST}\ntrain: images/train\nval: images/{val}\nnc: 1\nnames: ['plate']\n")
    return str(p)

DATA_REAL, DATA_PUBLIC = make_yaml("val_real"), make_yaml("val_public")
print(dict(n))
assert n["train_real"] and n["val_real"] and n["val_public"]

## 3. Mốc so sánh: model gốc trên val ảnh thật

In [ ]:
from ultralytics import YOLO

if not BASE_MODEL:
    cands = sorted(p for p in INPUT.rglob("*.pt") if "plate" in p.name.lower()) if INPUT.exists() else []
    BASE_MODEL = str(cands[0]) if cands else "yolo11n.pt"
print("Model gốc:", BASE_MODEL)

def score(model_path, data):
    r = YOLO(model_path).val(data=data, imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    return {"P": round(float(r.box.mp), 4), "R": round(float(r.box.mr), 4),
            "mAP50": round(float(r.box.map50), 4), "mAP50-95": round(float(r.box.map), 4)}

baseline = {"real": score(BASE_MODEL, DATA_REAL), "public": score(BASE_MODEL, DATA_PUBLIC)} if BASE_MODEL != "yolo11n.pt" else None
print("Mốc:", baseline)

## 4. Train (tự dừng sau `TRAIN_HOURS` giờ; `best.pt` theo mAP val ảnh thật)

In [ ]:
RUN = WORK / "runs" / "plate_real"
last = RUN / "weights" / "last.pt"

if RESUME and last.exists():
    print("Train tiếp từ", last)
    try:
        YOLO(str(last)).train(resume=True)
    except Exception as e:  # last.pt đã chạy hết thời gian -> dùng kết quả cũ
        print("Không resume được:", e)
else:
    YOLO(BASE_MODEL).train(
        data=DATA_REAL,          # val = ảnh camera thật -> best.pt chọn theo đúng thứ cần
        epochs=1000,             # trần; thực tế chặn bởi time=
        time=TRAIN_HOURS,
        patience=40,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        workers=4,
        cache="ram",
        project=str(RUN.parent),
        name=RUN.name,
        exist_ok=True,
        seed=42,
        plots=True,
        close_mosaic=10,
    )

## 5. So sánh với model gốc + xuất file

In [ ]:
best = RUN / "weights" / "best.pt"
assert best.exists(), f"Không có {best} — xem log ô train phía trên"

result = {"real": score(str(best), DATA_REAL), "public": score(str(best), DATA_PUBLIC)}
print(f"{'':6s}{'val':8s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
for name, res in (("gốc", baseline), ("MỚI", result)):
    if res:
        for split in ("real", "public"):
            m = res[split]
            print(f"{name:6s}{split:8s}{m['P']:8.3f}{m['R']:8.3f}{m['mAP50']:8.3f}{m['mAP50-95']:10.3f}")

shutil.copy2(best, WORK / "plate_real_best.pt")
(WORK / "plate_real_metrics.json").write_text(json.dumps(
    {"base": BASE_MODEL, "baseline": baseline, "new": result, "real_meta": meta}, indent=1))
shutil.make_archive(str(WORK / "plate_real_run"), "zip", RUN)
for d in (RAW, DST, WORK / "real_raw"):  # dọn bớt Output cho dễ tải
    shutil.rmtree(d, ignore_errors=True)
for f in sorted(WORK.glob("plate_real_*")):
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

## 6. Đưa về máy

1. Tab **Output** → tải `plate_real_best.pt` (+ `plate_real_metrics.json`).
2. Lưu thành `models/plate_real_best.pt` — **đừng ghi đè** `plate_best.pt` (release manifest + test khoá hash file đó).
3. So với `plate_best.pt` trên video thật **không có trong train** (v3, v5, v13 và `2026-10-01 18-33-23.mp4`), xem tận mắt các box chỉ 1 bên bắt được. Hơn thật thì bật bằng env `PLATE_MODEL_PATH=models/plate_real_best.pt`.

Lưu ý: val ảnh thật chỉ có 66 biển ⇒ mAP dao động vài điểm là bình thường; đừng chọn model chỉ vì hơn 0.01.